# 📊 Sales Forecasting & Analytics — Exploratory Data Analysis (EDA)

This notebook covers the exploratory data analysis, data cleaning, feature engineering, and model validation process for the **Sales Forecasting & Analytics** system. We use the public **Sample Superstore Sales** dataset.

--- 
## 1. Environment Setup & Data Acquisition
We will load libraries and import the data downloading and cleaning utilities from our project engine (`model.py`).

In [ ]:
import os
import sys
# Ensure project root is in python path
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from model import download_data, load_and_clean_data, aggregate_daily_sales, engineer_features, train_and_evaluate_models

# Style config for matplotlib
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

In [ ]:
# Download the dataset from GitHub source if missing
DATA_PATH = "../data/sales.csv"
download_data(path=DATA_PATH)

--- 
## 2. Load and Clean Dataset
We'll examine the raw columns, map them to standardized column names, convert date fields, drop duplicates, handle nulls, and cap outliers.

In [ ]:
# Read the raw data
raw_df = pd.read_csv(DATA_PATH)
print("Raw Data Shape:", raw_df.shape)
print("\nRaw Columns:", list(raw_df.columns))
raw_df.head(3)

In [ ]:
# Clean data using load_and_clean_data script from model.py
cleaned_df = load_and_clean_data(path=DATA_PATH)
print("\nCleaned Columns:", list(cleaned_df.columns))
cleaned_df.head(3)

--- 
## 3. Exploratory Data Analysis (EDA)
Let's review the main metrics and plot general statistics: Sales over time, Category distribution, Region, and Seasonality.

In [ ]:
# Calculate high-level metrics
total_revenue = cleaned_df['Sales'].sum()
total_quantity = cleaned_df['Quantity'].sum()
average_sale = cleaned_df['Sales'].mean()
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Products Sold: {total_quantity:,}")
print(f"Average Transaction Sale: ${average_sale:,.2f}")

In [ ]:
# Visualizing Sales distribution by Category
cat_sales = cleaned_df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
plt.figure(figsize=(8, 5))
plt.pie(cat_sales, labels=cat_sales.index, autopct='%1.1f%%', colors=['#6366F1', '#A855F7', '#EC4899'], startangle=140, explode=[0.05, 0.05, 0.05])
plt.title('Sales share by Category')
plt.show()

In [ ]:
# Visualizing Top 10 selling Products
top_products = cleaned_df.groupby('Product')['Sales'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=top_products.values, y=top_products.index, palette='plasma')
plt.title('Top 10 Selling Products by Revenue')
plt.xlabel('Revenue ($)')
plt.ylabel('Product Name')
plt.show()

In [ ]:
# Aggregating daily data
daily_df = aggregate_daily_sales(cleaned_df)
print("Daily Aggregated Data Shape:", daily_df.shape)

# Historical Sales Trend with 30-Day moving average
plt.figure(figsize=(15, 6))
plt.plot(daily_df['Date'], daily_df['Sales'], label='Daily Sales', alpha=0.3, color='indigo')
plt.plot(daily_df['Date'], daily_df['Sales'].rolling(window=30).mean(), label='30-Day Moving Avg', color='deeppink', linewidth=2)
plt.title('Historical Daily Sales & 30-Day Rolling Average')
plt.xlabel('Date')
plt.ylabel('Revenue ($)')
plt.legend()
plt.show()

--- 
## 4. Feature Engineering
For predictive modeling, we extract date/calendar features from the daily aggregated dates and calculate lag and rolling features.

In [ ]:
# Create daily features
features_df = engineer_features(daily_df)
print("Features DataFrame Shape:", features_df.shape)
features_df.head(3)

In [ ]:
# Analyze correlation between engineered features and Target (Sales)
correlation_matrix = features_df.drop(columns=['Date']).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.show()

--- 
## 5. Model Training & Evaluation
We evaluate Linear Regression, Random Forest, and XGBoost using chronological splits (first 80% train, last 20% test).

In [ ]:
# Train and evaluate
models, metrics_df, predictions_df = train_and_evaluate_models(features_df)
print("Model Evaluation Summary:")
metrics_df

In [ ]:
# Plot test-set predictions vs actual sales
plt.figure(figsize=(15, 6))
plt.plot(predictions_df['Date'], predictions_df['Actual'], label='Actual Sales', alpha=0.5, color='gray')
for model_name in ['Linear Regression', 'Random Forest', 'XGBoost']:
    plt.plot(predictions_df['Date'], predictions_df[model_name], label=f'{model_name} Prediction', alpha=0.8)
plt.title('Model Predictions vs Actual Sales on Test Set')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.show()

### Conclusion & Insights
- Calendar features and lags provide strong predictors for capturing seasonal variation.
- Machine Learning regression models (XGBoost/Random Forest) generally capture complex seasonal spikes better than simple Linear Regression.
- Moving averages (7-day and 30-day) help smooth out daily variance and provide standard references for baseline predictions.